[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philmui/ai-foundations/blob/main/06_cleaning_transformation/tutorial.ipynb)

## ▶️ Run this notebook in Google Colab

**Option A — One click (recommended):** Click the **Open in Colab** badge above. It opens this notebook directly in a free cloud runtime — nothing to install locally.

**Option B — Upload manually:**
1. Go to [colab.research.google.com](https://colab.research.google.com).
2. Choose **File → Upload notebook** and select this `tutorial.ipynb`.

**Then, in Colab:**
1. Run the **first code cell** (the dependency-setup cell) — it installs everything this notebook needs directly into the Colab kernel. No `pip install`, `uv sync`, or `pyproject.toml` required.
2. Run the remaining cells top to bottom via **Runtime → Run all** (or `Ctrl/Cmd+F9`).

> 💡 This notebook is **fully self-contained**: any data files it uses are generated inside the notebook, so it runs end-to-end in Colab with no external files.

---

In [ ]:
# === Inline dependency setup (self-contained; no `uv sync` / pyproject.toml needed) ===
# Installs this notebook's libraries directly into the running kernel.
# Works in local Jupyter, VS Code, and Google Colab. Safe to re-run (idempotent).
import sys, subprocess

_DEPS = [
    'python-dotenv>=1.0.0',
    'numpy>=1.26.0',
    'pandas>=2.1.0',
    'matplotlib>=3.7',
]

# Ensure pip is available in this kernel (some minimal venvs ship without it), then install.
try:
    import pip  # noqa: F401
except ModuleNotFoundError:
    import ensurepip; ensurepip.bootstrap()

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_DEPS])
print('\u2713 Dependencies installed:', ', '.join(_DEPS))


# Module 6: Data Cleaning & Transformation

**AI Research Foundations Minicourse**

---

## 1. Introduction: Garbage In, Garbage Out

You've probably heard the phrase "garbage in, garbage out" — it's especially true in AI and machine learning. No matter how sophisticated your model is, if you feed it dirty, inconsistent, or incomplete data, you'll get unreliable results.

**Real-world data is messy:**
- Missing values
- Duplicate entries
- Inconsistent formatting
- Invalid values
- Wrong data types

> **Key Insight:** Data scientists and ML engineers spend approximately **80% of their time** on data cleaning and preparation, and only 20% on actual modeling. Mastering data cleaning is essential for success in AI.

In this module, you'll learn how to:
1. Inspect datasets to identify quality issues
2. Remove duplicates
3. Handle missing values strategically
4. Filter data using boolean indexing
5. Fix formatting and standardize values
6. Correct data types
7. Split data into train and test sets
8. Export clean datasets for downstream use

```mermaid
flowchart LR
    A[Raw Data] --> B[Inspect]
    B --> C[Deduplicate]
    C --> D[Handle Missing]
    D --> E[Fix Formatting]
    E --> F[Filter Invalid]
    F --> G[Fix Types]
    G --> H[Split Train/Test]
    H --> I[Export Clean Data]
    
    style A fill:#f9e79f
    style I fill:#abebc6
```

**The Cleaning Pipeline:** Every step matters. Skip one, and your model may crash or produce wrong predictions.

In [ ]:
# Setup: Load environment variables and import libraries
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', None)

print("✓ Environment loaded")
print(f"✓ Pandas version: {pd.__version__}")

---

## 2. Inspecting Dirty Data

Before cleaning, we need to understand what problems exist. Think of this like a doctor's diagnosis — you can't prescribe treatment without examining the patient first.

Our dataset contains prompt-response pairs from various AI models (GPT-4, Claude-3, etc.) with metadata like temperature, token usage, and ratings.

**Key inspection techniques:**
- `.info()` — data types, memory usage, non-null counts
- `.isna().sum()` — count missing values per column
- `.duplicated().sum()` — count duplicate rows
- `.head()` and `.sample()` — visual inspection of actual data

In [ ]:
# --- Self-contained data setup (creates data/dirty_dataset.csv for Colab) ---
from pathlib import Path
Path('data').mkdir(exist_ok=True)
_CSV_DATA = r'''prompt_id,prompt_text,response_text,model,temperature,tokens_used,rating,category,date
1,Write a haiku about AI,   Circuits pulse and think,GPT-4,0.7,45,4.2,creative,2024-01-15
2,Explain quantum computing,Quantum mechanics uses superposition...,gpt-4,0.5,234,4.8,technical,01/16/2024
3,Translate 'hello' to Spanish,Hola,Claude-3,0.3,12,5.0,language,2024-01-17
4,Write a haiku about AI,   Circuits pulse and think,GPT-4,0.7,45,4.2,creative,2024-01-15
5,Summarize climate change,Climate change refers to long-term...,GPT4,0.6,,4.5,science,2024-01-18
6,Debug this Python code,The error is on line 3...,Claude-3-Opus,0.8,156,4.9,coding,01/19/2024
7,,Machine learning is a subset of AI...,GPT-3.5,0.5,89,3.8,technical,2024-01-20
8,What is photosynthesis?,Plants convert light energy...,gpt-4,0.4,123,4.6,science,2024-01-21
9,Recommend a book,I recommend 'Sapiens' by...,claude-3,2.5,67,4.1,general,2024-01-22
10,Write a poem about nature,,GPT-4,0.9,0,3.5,creative,01/23/2024
11,Explain blockchain,Blockchain is a distributed ledger...,GPT-4,0.5,198,4.7,technical,2024-01-24
12,Translate 'goodbye' to French,Au revoir,Claude3,0.3,14,5.0,language,2024-01-25
13,What is machine learning?,ML is a method of data analysis...,   gpt-4   ,0.6,145,4.4,technical,01/26/2024
14,Calculate 15% tip on $50,The tip would be $7.50,GPT-3.5,0.2,28,4.0,math,2024-01-27
15,Write a haiku about AI,   Circuits pulse and think,GPT-4,0.7,45,4.2,creative,2024-01-15
16,Explain neural networks,Neural networks are computing systems...,Claude-3,0.7,267,4.9,technical,2024-01-28
17,Translate 'thank you' to Japanese,Arigato,gpt4,0.3,15,4.8,language,01/29/2024
18,Summarize World War II,,GPT-4,0.6,0,3.2,history,2024-01-30
19,Debug Java code,The issue is with the null pointer...,Claude-3-Opus,0.8,189,4.7,coding,2024-01-31
20,What is DNA?,,GPT-3.5,0.5,134,4.3,science,02/01/2024
21,Recommend a restaurant,Try the Italian place on 5th...,gpt-4,0.7,52,4.0,general,2024-02-02
22,Write a story about robots,In a future where robots...,GPT-4,0.9,312,4.6,creative,2024-02-03
23,Explain relativity,Einstein's theory describes...,Claude-3,0.6,256,4.9,science,02/04/2024
24,Translate 'good morning' to German,Guten Morgen,GPT4,0.3,16,5.0,language,2024-02-05
25,What is recursion?,Recursion is when a function calls itself...,gpt-4,0.5,98,4.5,coding,2024-02-06
26,Calculate compound interest,Using the formula A = P(1+r/n)^(nt)...,GPT-3.5,0.2,145,4.2,math,02/07/2024
27,,The mitochondria is the powerhouse...,Claude-3,0.5,87,4.1,science,2024-02-08
28,Debug SQL query,Missing JOIN clause between tables...,gpt-4,0.8,176,4.8,coding,2024-02-09
29,Write a poem about space,Beyond the atmosphere so vast...,GPT-4,0.9,234,4.4,creative,02/10/2024
30,Explain cryptocurrency,Digital currency using cryptography...,Claude3,0.6,223,4.6,technical,2024-02-11
31,Translate 'I love you' to Italian,Ti amo,GPT-4,0.3,13,5.0,language,2024-02-12
32,What is evolution?,Evolution is the change in heritable...,gpt-4,0.5,198,4.7,science,02/13/2024
33,Recommend a movie,   Check out 'Inception'...,GPT-3.5,0.7,45,3.9,general,2024-02-14
34,Write a haiku about code,Functions compile clean...,Claude-3,0.7,42,4.3,creative,2024-02-15
35,Explain data structures,Arrays store elements in contiguous...,GPT-4,0.6,287,4.8,coding,02/16/2024
36,Calculate mortgage payment,,gpt-4,0.2,0,3.5,math,2024-02-17
37,What is entropy?,Entropy measures disorder in a system...,Claude-3-Opus,0.5,167,4.6,science,2024-02-18
38,Debug React component,The state is not updating because...,GPT4,0.8,198,4.9,coding,02/19/2024
39,Write a story about time travel,Sarah stepped into the machine...,GPT-4,0.9,445,4.5,creative,2024-02-20
40,Explain cloud computing,Cloud computing delivers computing services...,claude-3,0.6,234,4.7,technical,2024-02-21
41,Translate 'beautiful' to French,Belle,GPT-3.5,0.3,12,4.8,language,02/22/2024
42,What is gravity?,Gravity is a force that attracts...,gpt-4,0.4,123,4.5,science,2024-02-23
43,Recommend a podcast,,Claude-3,0.7,0,3.7,general,2024-02-24
44,Write a poem about music,Melodies drift through the air...,GPT-4,0.9,187,4.4,creative,02/25/2024
45,Explain algorithms,An algorithm is a step-by-step procedure...,   GPT4   ,0.6,212,4.8,coding,2024-02-26
46,Calculate standard deviation,Using the formula sqrt(Σ(x-μ)²/n)...,GPT-3.5,0.2,234,4.3,math,02/27/2024
47,What is osmosis?,Osmosis is the movement of water...,Claude-3,0.5,134,4.6,science,2024-02-28
48,Debug Python function,   The loop index is off by one...,gpt-4,0.8,145,4.7,coding,2024-02-29
49,Write a haiku about ocean,Waves crash on the shore...,GPT-4,0.7,43,4.2,creative,03/01/2024
50,Explain APIs,An API is an interface that allows...,Claude3,0.6,198,4.9,technical,2024-03-02
51,Translate 'friend' to Spanish,Amigo,gpt-4,0.3,11,5.0,language,03/03/2024
52,What is mitosis?,Mitosis is cell division that produces...,GPT-4,0.5,176,4.7,science,2024-03-04
53,Recommend a video game,Try 'Elden Ring' if you like...,GPT-3.5,0.7,67,4.1,general,03/05/2024
54,Write a story about dragons,The ancient dragon stirred...,Claude-3,0.9,389,4.6,creative,2024-03-06
55,Explain databases,,gpt-4,0.6,0,3.8,technical,2024-03-07
56,Calculate probability,"Using combinations: C(n,k) = n!/(k!(n-k)!)...",GPT4,0.2,198,4.4,math,03/08/2024
57,What is metabolism?,Metabolism is the set of chemical reactions...,Claude-3-Opus,0.5,145,4.6,science,2024-03-09
58,Debug CSS layout,The flexbox container needs...,gpt-4,0.8,123,4.8,coding,03/10/2024
59,Write a poem about winter,Snowflakes fall so softly...,GPT-4,0.9,156,4.3,creative,2024-03-11
60,Explain encryption,Encryption converts plaintext to ciphertext...,   claude-3   ,0.6,267,4.9,technical,03/12/2024
61,Translate 'water' to Chinese,水 (shuǐ),GPT-3.5,0.3,18,4.7,language,2024-03-13
62,What is photosynthesis?,Plants convert light energy...,gpt-4,0.4,123,4.6,science,2024-01-21
63,Recommend a TV show,Watch 'Breaking Bad' for...,GPT-4,0.7,54,4.2,general,03/14/2024
64,,Version control tracks changes to files...,Claude-3,0.6,189,4.5,coding,2024-03-15
65,Calculate derivative,Using power rule: d/dx(x^n) = nx^(n-1)...,gpt-4,0.2,167,4.6,math,03/16/2024
66,What is immunity?,The immune system defends against pathogens...,GPT4,0.5,198,4.7,science,2024-03-17
67,Debug JavaScript async,The promise chain is missing a catch...,Claude-3-Opus,0.8,234,4.9,coding,03/18/2024
68,Write a haiku about rain,Droplets tap the glass...,GPT-4,0.7,41,4.1,creative,2024-03-19
69,Explain virtualization,Virtualization creates virtual versions...,gpt-4,0.6,245,4.8,technical,03/20/2024
70,Translate 'peace' to Arabic,سلام (salām),Claude3,0.3,19,4.9,language,2024-03-21
71,What is friction?,Friction is a force that opposes motion...,GPT-3.5,0.4,134,4.4,science,03/22/2024
72,Recommend a restaurant,Try the Italian place on 5th...,gpt-4,0.7,52,4.0,general,2024-02-02
73,Write a story about magic,In a world where magic flows...,GPT-4,3.0,412,4.7,creative,2024-03-23
74,Explain microservices,Microservices architecture structures...,Claude-3,0.6,289,4.8,technical,03/24/2024
75,Calculate integral,Using substitution method: ∫u du...,gpt-4,0.2,178,4.5,math,2024-03-25
76,,Photosynthesis occurs in chloroplasts...,GPT4,0.5,0,3.9,science,2024-03-26
77,Debug TypeScript types,The interface needs to extend...,Claude-3-Opus,0.8,156,4.9,coding,03/27/2024
78,Write a poem about dreams,In slumber's gentle embrace...,GPT-4,0.9,198,4.4,creative,2024-03-28
79,Explain containerization,Containers package applications with...,gpt-4,0.6,256,4.9,technical,03/29/2024
80,Translate 'hope' to Korean,희망 (huimang),Claude-3,0.3,17,4.8,language,2024-03-30
'''
Path('data/dirty_dataset.csv').write_text(_CSV_DATA)
print('✓ Sample dataset written to data/dirty_dataset.csv')

In [ ]:
# Load the dirty dataset
df = pd.read_csv('data/dirty_dataset.csv')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")

# Display first few rows
df.head(10)

In [ ]:
# Get overview of data types and non-null counts
print("=== Dataset Info ===")
df.info()
print("\n" + "="*50)

In [ ]:
# Count missing values per column
missing_counts = df.isna().sum()
print("=== Missing Values ===")
print(missing_counts)
print(f"\nTotal missing values: {missing_counts.sum()}")
print(f"Missing value percentage: {(missing_counts.sum() / (df.shape[0] * df.shape[1]) * 100):.2f}%")

In [ ]:
# Count duplicate rows
num_duplicates = df.duplicated().sum()
print(f"=== Duplicate Rows ===")
print(f"Number of duplicate rows: {num_duplicates}")
print(f"Percentage: {(num_duplicates / len(df) * 100):.2f}%")

# Show some duplicate examples
if num_duplicates > 0:
    print("\nExample duplicates:")
    duplicate_mask = df.duplicated(keep=False)
    print(df[duplicate_mask].sort_values('prompt_id').head(6))

In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(10, 6))
missing_data = df.isna().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

bars = ax.bar(range(len(missing_data)), missing_data.values, color='#e74c3c')
ax.set_xticks(range(len(missing_data)))
ax.set_xticklabels(missing_data.index, rotation=45, ha='right')
ax.set_ylabel('Number of Missing Values', fontsize=12)
ax.set_xlabel('Column', fontsize=12)
ax.set_title('Missing Values by Column (Before Cleaning)', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\n📊 Visualization shows which columns have the most missing data.")

### Inspection Summary

From our inspection, we've identified:
1. **Missing values** in `prompt_text`, `response_text`, `tokens_used`, and `rating`
2. **Duplicate rows** (exact copies)
3. **Formatting issues** likely in `model` names and `date` formats
4. **Data type issues** — dates stored as strings, not datetime objects

> **Key Insight:** Always inspect your data before cleaning. Understanding the problems helps you choose the right cleaning strategy.

---

## 3. Removing Duplicates

Duplicate rows are exact copies of other rows. They can occur due to:
- Data collection errors
- Multiple system logs of the same event
- Database synchronization issues

**Why duplicates are dangerous in ML:**
- **Data leakage:** If the same sample appears in both training and test sets, you're testing on data the model has already seen
- **Biased models:** Duplicates give certain patterns more weight
- **Inflated metrics:** Your accuracy looks better than it really is

**Key functions:**
- `.duplicated()` — returns boolean Series marking duplicates
- `.drop_duplicates()` — removes duplicate rows
- `keep='first'` — keeps the first occurrence (default)
- `keep='last'` — keeps the last occurrence
- `keep=False` — marks all duplicates (including the first)
- `subset=['col1', 'col2']` — check duplication only on specific columns

In [ ]:
# Find duplicates (complete row matches)
duplicate_mask = df.duplicated(keep=False)
print(f"Total rows with duplicates: {duplicate_mask.sum()}")

# Show duplicate examples
print("\nDuplicate rows:")
df[duplicate_mask].sort_values(['prompt_id', 'prompt_text']).head(10)

In [ ]:
# Remove duplicates (keep first occurrence)
df_clean = df.drop_duplicates(keep='first')

print(f"Original rows: {len(df)}")
print(f"After deduplication: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} duplicate rows")

In [ ]:
# Example: Subset duplicates (checking only prompt_text and model)
# This finds prompts that are the same but may have different responses
subset_duplicates = df_clean.duplicated(subset=['prompt_text', 'model'], keep=False)
print(f"Rows with duplicate prompt+model combinations: {subset_duplicates.sum()}")

if subset_duplicates.sum() > 0:
    print("\nExample: Same prompt sent to same model multiple times:")
    print(df_clean[subset_duplicates].sort_values(['prompt_text', 'model'])[['prompt_text', 'model', 'temperature', 'rating']].head(10))

> **Key Insight:** Deduplication is not always "delete everything that matches." Sometimes you want to check duplication on only certain columns (e.g., user IDs, timestamps) while allowing other fields to vary.

---

## 4. Handling Missing Values

Missing values (NaN, None, empty strings) are inevitable in real-world data. The question is: **What do you do about them?**

Two main strategies:

1. **Remove (`.dropna()`)** — Delete rows or columns with missing values
   - **Pros:** Clean, no assumptions made
   - **Cons:** Lose potentially valuable data
   - **When to use:** When missing data is rare (<5%) or critical columns are empty

2. **Fill (`.fillna()`)** — Replace missing values with a substitute
   - **Pros:** Keep all rows, maintain sample size
   - **Cons:** Introduces assumptions, may add noise
   - **When to use:** When missing data is common or rows are valuable

```mermaid
flowchart TD
    A[Missing Value Detected] --> B{Is it critical<br/>information?}
    B -->|Yes| C[Drop row]
    B -->|No| D{High missing %<br/>in column?}
    D -->|Yes >50%| E[Drop column]
    D -->|No| F{What type?}
    F -->|Numeric| G[Fill with mean/median]
    F -->|Categorical| H[Fill with mode/'unknown']
    F -->|Text| I[Fill with empty string]
    
    style C fill:#e74c3c
    style E fill:#e74c3c
    style G fill:#3498db
    style H fill:#3498db
    style I fill:#3498db
```

**Common fill strategies:**
- **Numeric:** Mean, median, 0, -1 (as a flag)
- **Categorical:** Mode (most common), "unknown", "missing"
- **Text:** Empty string, "N/A"

In [ ]:
# Current state of missing values
print("=== Missing Values Before Handling ===")
print(df_clean.isna().sum())
print(f"\nTotal rows: {len(df_clean)}")

In [ ]:
# Strategy 1: Drop rows where critical columns are missing
# prompt_text is critical — we can't use a sample without it
print("Dropping rows with missing prompt_text...")
df_clean = df_clean.dropna(subset=['prompt_text'])
print(f"Rows remaining: {len(df_clean)}")

In [ ]:
# Strategy 2: Fill missing response_text with a placeholder
# Empty responses might indicate failures, but we want to keep the metadata
print("Filling missing response_text...")
df_clean['response_text'] = df_clean['response_text'].fillna('[No response generated]')
print(f"response_text missing values: {df_clean['response_text'].isna().sum()}")

In [ ]:
# Strategy 3: Fill missing tokens_used with 0
# 0 makes sense — if no response, no tokens were used
print("Filling missing tokens_used with 0...")
df_clean['tokens_used'] = df_clean['tokens_used'].fillna(0)
print(f"tokens_used missing values: {df_clean['tokens_used'].isna().sum()}")

In [ ]:
# Strategy 4: Fill missing rating with median
# Median is robust to outliers
median_rating = df_clean['rating'].median()
print(f"Median rating: {median_rating}")
print("Filling missing rating with median...")
df_clean['rating'] = df_clean['rating'].fillna(median_rating)
print(f"rating missing values: {df_clean['rating'].isna().sum()}")

In [ ]:
# Verify: Check final missing value counts
print("\n=== Missing Values After Handling ===")
print(df_clean.isna().sum())
print(f"\nTotal missing values remaining: {df_clean.isna().sum().sum()}")

> **Key Insight:** The strategy you choose depends on context. Dropping might be acceptable for 1% missing data but disastrous for 30%. Filling with mean makes sense for exam scores but not for binary flags. Always think about what the missing value *means* before deciding.

---

## 5. Boolean Indexing for Filtering

Boolean indexing lets you filter rows based on conditions. This is essential for removing invalid data or keeping only samples that meet quality criteria.

**How it works:**
1. Create a **boolean mask** — a Series of True/False values
2. Use the mask to index the DataFrame
3. Only rows where the mask is `True` are kept

**Operators:**
- `==` equal
- `!=` not equal
- `>`, `<`, `>=`, `<=` comparisons
- `&` AND (both conditions must be True)
- `|` OR (at least one condition must be True)
- `~` NOT (inverts True/False)

**Important:** When combining conditions, always use parentheses: `(condition1) & (condition2)`

In [ ]:
# Example 1: Simple boolean mask
# Find rows where temperature > 1.0 (suspicious — typical range is 0.0-1.0)
high_temp_mask = df_clean['temperature'] > 1.0
print("Rows with temperature > 1.0:")
print(df_clean[high_temp_mask][['prompt_id', 'model', 'temperature', 'category']])

In [ ]:
# Example 2: Combining conditions with AND (&)
# Find high-quality coding samples: category='coding' AND rating >= 4.5
quality_coding_mask = (df_clean['category'] == 'coding') & (df_clean['rating'] >= 4.5)
print(f"High-quality coding samples: {quality_coding_mask.sum()}")
print("\nExamples:")
print(df_clean[quality_coding_mask][['prompt_text', 'model', 'rating', 'category']].head())

In [ ]:
# Example 3: Combining conditions with OR (|)
# Find samples that are either creative OR have high tokens
creative_or_long_mask = (df_clean['category'] == 'creative') | (df_clean['tokens_used'] > 200)
print(f"Creative or long responses: {creative_or_long_mask.sum()}")

In [ ]:
# Example 4: NOT operator (~) to invert
# Find samples that are NOT technical
not_technical_mask = ~(df_clean['category'] == 'technical')
print(f"Non-technical samples: {not_technical_mask.sum()}")

In [ ]:
# Apply filter: Remove rows with invalid temperature (> 1.0)
print(f"Rows before filtering temperature: {len(df_clean)}")
df_clean = df_clean[df_clean['temperature'] <= 1.0]
print(f"Rows after filtering temperature: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} rows with invalid temperature")

In [ ]:
# Apply filter: Keep only rows with reasonable token counts (> 0)
print(f"Rows before filtering tokens: {len(df_clean)}")
df_clean = df_clean[df_clean['tokens_used'] > 0]
print(f"Rows after filtering tokens: {len(df_clean)}")
print(f"Kept only rows with token usage > 0")

> **Key Insight:** Boolean indexing is powerful but requires careful thinking. Always ask: "What business logic justifies this filter?" Removing valid data is worse than keeping a few outliers.

---

## 6. Fixing Formatting Issues

Real-world data often has inconsistent formatting:
- **Whitespace:** `"  GPT-4  "` vs `"GPT-4"`
- **Case:** `"GPT-4"` vs `"gpt-4"`
- **Abbreviations:** `"GPT-4"` vs `"GPT4"`

These look the same to humans but are different to computers. Inconsistent formatting causes:
- Incorrect groupings
- Inflated unique value counts
- ML models treating similar values as different categories

**String methods for cleaning:**
- `.str.strip()` — remove leading/trailing whitespace
- `.str.lower()` — convert to lowercase
- `.str.upper()` — convert to uppercase
- `.str.replace()` — replace patterns
- `.str.contains()` — check for substring
- `.map()` — apply custom mappings
- `.replace()` — replace exact matches

In [ ]:
# Inspect current formatting issues
print("=== Unique model names (before cleaning) ===")
print(df_clean['model'].unique())
print(f"\nNumber of unique models: {df_clean['model'].nunique()}")

In [ ]:
# Step 1: Strip whitespace
df_clean['model'] = df_clean['model'].str.strip()
print("After stripping whitespace:")
print(df_clean['model'].unique())

In [ ]:
# Step 2: Standardize to lowercase for consistency
df_clean['model'] = df_clean['model'].str.lower()
print("After converting to lowercase:")
print(df_clean['model'].unique())

In [ ]:
# Step 3: Standardize abbreviations with mapping
# Map all variations to standard names
model_mapping = {
    'gpt4': 'gpt-4',
    'gpt-4': 'gpt-4',
    'gpt-3.5': 'gpt-3.5',
    'claude3': 'claude-3',
    'claude-3': 'claude-3',
    'claude-3-opus': 'claude-3-opus'
}

df_clean['model'] = df_clean['model'].replace(model_mapping)
print("After standardizing abbreviations:")
print(df_clean['model'].unique())
print(f"\nNumber of unique models: {df_clean['model'].nunique()}")

In [ ]:
# Example: Fix formatting in category column
print("=== Category formatting ===")
print("Before:")
print(df_clean['category'].unique())

# Strip and lowercase
df_clean['category'] = df_clean['category'].str.strip().str.lower()
print("\nAfter:")
print(df_clean['category'].unique())

> **Key Insight:** Standardization reduces noise. Instead of treating "GPT-4", "gpt-4", and "GPT4" as three different models, we recognize they're the same. This improves grouping accuracy and reduces model complexity.

---

## 7. Data Type Corrections

Data loaded from CSV files often has incorrect types:
- Dates stored as strings: `"2024-01-15"` instead of `datetime`
- Numbers stored as strings: `"123"` instead of `123`
- Mixed types: Some dates as `MM/DD/YYYY`, others as `YYYY-MM-DD`

**Why types matter:**
- **Performance:** Operations on correct types are faster
- **Functionality:** Can't do date math on strings
- **Memory:** Proper types use less memory
- **ML models:** Most algorithms require numeric inputs

**Key functions:**
- `pd.to_datetime()` — convert to datetime
- `pd.to_numeric()` — convert to numeric (int or float)
- `.astype()` — explicit type conversion
- `errors='coerce'` — turn invalid values into NaN instead of crashing

In [ ]:
# Check current data types
print("=== Current Data Types ===")
print(df_clean.dtypes)
print("\nNotice: 'date' is 'object' (string), not 'datetime'")

In [ ]:
# Inspect date formats
print("Sample dates:")
print(df_clean['date'].head(10).tolist())
print("\nNotice: Mixed formats!")
print("- Format 1: YYYY-MM-DD (e.g., '2024-01-15')")
print("- Format 2: MM/DD/YYYY (e.g., '01/16/2024')")

In [ ]:
# Convert date column to datetime
# pd.to_datetime() is smart — it can handle multiple formats
df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')

print("After conversion:")
print(df_clean['date'].dtype)
print("\nSample dates:")
print(df_clean['date'].head(10))

# Check if any dates failed to convert
failed_dates = df_clean['date'].isna().sum()
print(f"\nFailed conversions: {failed_dates}")

In [ ]:
# Ensure tokens_used is integer
df_clean['tokens_used'] = df_clean['tokens_used'].astype(int)
print(f"tokens_used type: {df_clean['tokens_used'].dtype}")

In [ ]:
# Verify final types
print("=== Final Data Types ===")
print(df_clean.dtypes)

In [ ]:
# Now we can do date operations!
# Extract month and day of week
df_clean['month'] = df_clean['date'].dt.month
df_clean['day_of_week'] = df_clean['date'].dt.day_name()

print("Example: Date-based features")
print(df_clean[['date', 'month', 'day_of_week']].head())

print("\nDistribution by month:")
print(df_clean['month'].value_counts().sort_index())

> **Key Insight:** Proper data types unlock functionality. Once dates are datetime objects, you can extract month, year, day of week, calculate differences, and filter by date ranges — none of which work on strings.

---

## 8. Train/Test Split

In machine learning, we **never** test on the same data we trained on. That's like studying with the exam questions beforehand — your score looks great but doesn't reflect real performance.

**Why split data:**
1. **Prevent overfitting:** Model memorizes training data but fails on new data
2. **Honest evaluation:** Test set simulates real-world performance
3. **Detect problems:** Gap between train and test scores reveals issues

**Common splits:**
- **80/20:** 80% training, 20% testing (most common)
- **70/30:** More test data for small datasets
- **90/10:** More training data when data is plentiful

**Important considerations:**
- **Random sampling:** Use `.sample()` to randomly select rows
- **Reproducibility:** Set `random_state` so split is consistent
- **Class balance:** Check that both splits have similar distributions

```mermaid
pie title Train/Test Split (80/20)
    "Training Set (80%)" : 80
    "Test Set (20%)" : 20
```

In [ ]:
# Set random seed for reproducibility
RANDOM_STATE = 42

# Shuffle the data
df_shuffled = df_clean.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"Total samples: {len(df_shuffled)}")

In [ ]:
# Calculate split index (80% train, 20% test)
train_size = int(0.8 * len(df_shuffled))
print(f"Train size: {train_size} (80%)")
print(f"Test size: {len(df_shuffled) - train_size} (20%)")

In [ ]:
# Split into train and test
train_df = df_shuffled.iloc[:train_size]
test_df = df_shuffled.iloc[train_size:]

print(f"Training set: {len(train_df)} rows")
print(f"Test set: {len(test_df)} rows")
print(f"Total: {len(train_df) + len(test_df)} rows")

In [ ]:
# Visualize the split
fig, ax = plt.subplots(figsize=(8, 6))
sizes = [len(train_df), len(test_df)]
labels = [f'Train\n{len(train_df)} rows\n({len(train_df)/len(df_shuffled)*100:.1f}%)',
          f'Test\n{len(test_df)} rows\n({len(test_df)/len(df_shuffled)*100:.1f}%)']
colors = ['#3498db', '#e74c3c']
explode = (0.05, 0)

ax.pie(sizes, labels=labels, colors=colors, explode=explode,
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
ax.set_title('Train/Test Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Check class balance: Compare category distribution
print("=== Category Distribution ===")
print("\nTraining set:")
train_dist = train_df['category'].value_counts(normalize=True).sort_index()
print(train_dist)

print("\nTest set:")
test_dist = test_df['category'].value_counts(normalize=True).sort_index()
print(test_dist)

print("\n✓ Distributions should be similar for a good split")

In [ ]:
# Visualize category distribution comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Training set distribution
train_counts = train_df['category'].value_counts()
ax1.bar(range(len(train_counts)), train_counts.values, color='#3498db', alpha=0.7)
ax1.set_xticks(range(len(train_counts)))
ax1.set_xticklabels(train_counts.index, rotation=45, ha='right')
ax1.set_ylabel('Count', fontsize=11)
ax1.set_title('Training Set Category Distribution', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Test set distribution
test_counts = test_df['category'].value_counts()
ax2.bar(range(len(test_counts)), test_counts.values, color='#e74c3c', alpha=0.7)
ax2.set_xticks(range(len(test_counts)))
ax2.set_xticklabels(test_counts.index, rotation=45, ha='right')
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('Test Set Category Distribution', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visual check: Distributions should look similar")

> **Key Insight:** A good train/test split preserves the distribution of your target variable and important features. If your test set has very different characteristics than your training set, your model won't generalize well.

---

## 9. Exporting Clean Data

After all that work, we need to save our clean datasets for later use. CSV is the most common format for tabular data — it's human-readable, widely supported, and efficient for moderate-sized datasets.

**Key parameters for `.to_csv()`:**
- `index=False` — don't write row numbers (usually not needed)
- `encoding='utf-8'` — standard encoding for international characters
- `float_format='%.4f'` — control decimal precision

In [ ]:
# Create output directory if it doesn't exist
import os
os.makedirs('data/output', exist_ok=True)
print("✓ Output directory ready")

In [ ]:
# Drop temporary columns (month, day_of_week) that were just for exploration
train_export = train_df.drop(columns=['month', 'day_of_week'])
test_export = test_df.drop(columns=['month', 'day_of_week'])

print(f"Columns to export: {list(train_export.columns)}")

In [ ]:
# Export training set
train_export.to_csv('data/output/Train.csv', index=False, encoding='utf-8')
print("✓ Exported: data/output/Train.csv")
print(f"  Rows: {len(train_export)}")
print(f"  Columns: {len(train_export.columns)}")

In [ ]:
# Export test set
test_export.to_csv('data/output/Test.csv', index=False, encoding='utf-8')
print("✓ Exported: data/output/Test.csv")
print(f"  Rows: {len(test_export)}")
print(f"  Columns: {len(test_export.columns)}")

In [ ]:
# Before/After Statistics Comparison
print("\n" + "="*60)
print("BEFORE vs AFTER CLEANING COMPARISON")
print("="*60)

print("\n📊 DATASET SIZE:")
print(f"  Before: {len(df)} rows")
print(f"  After:  {len(df_shuffled)} rows")
print(f"  Removed: {len(df) - len(df_shuffled)} rows ({(len(df) - len(df_shuffled))/len(df)*100:.1f}%)")

print("\n📊 MISSING VALUES:")
before_missing = df.isna().sum().sum()
after_missing = df_shuffled.isna().sum().sum()
print(f"  Before: {before_missing} missing values")
print(f"  After:  {after_missing} missing values")
print(f"  Reduction: {before_missing - after_missing} ({(before_missing - after_missing)/before_missing*100:.1f}%)")

print("\n📊 DUPLICATES:")
before_dupes = df.duplicated().sum()
after_dupes = df_shuffled.duplicated().sum()
print(f"  Before: {before_dupes} duplicates")
print(f"  After:  {after_dupes} duplicates")

print("\n📊 UNIQUE MODELS:")
print(f"  Before: {df['model'].nunique()} unique model names")
print(f"  After:  {df_shuffled['model'].nunique()} unique model names")
print(f"  Standardized: {df['model'].nunique() - df_shuffled['model'].nunique()} variants")

print("\n📊 DATA QUALITY:")
print(f"  Valid temperature range: {(df_shuffled['temperature'] <= 1.0).all()}")
print(f"  All tokens > 0: {(df_shuffled['tokens_used'] > 0).all()}")
print(f"  Dates are datetime: {df_shuffled['date'].dtype == 'datetime64[ns]'}")

print("\n" + "="*60)

In [ ]:
# Visualize cleaning pipeline impact
steps = ['Original', 'Deduped', 'No Missing\nPrompts', 'Filtered\nTemperature', 'Filtered\nTokens']
row_counts = [
    len(df),
    len(df.drop_duplicates()),
    len(df.drop_duplicates().dropna(subset=['prompt_text'])),
    len(df.drop_duplicates().dropna(subset=['prompt_text'])[df.drop_duplicates().dropna(subset=['prompt_text'])['temperature'] <= 1.0]),
    len(df_shuffled)
]

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c', '#e67e22', '#f39c12', '#52be80', '#3498db']
bars = ax.bar(range(len(steps)), row_counts, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_xticks(range(len(steps)))
ax.set_xticklabels(steps, fontsize=11)
ax.set_ylabel('Number of Rows', fontsize=12)
ax.set_xlabel('Cleaning Step', fontsize=12)
ax.set_title('Impact of Each Cleaning Step on Dataset Size', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (bar, count) in enumerate(zip(bars, row_counts)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(count)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Chart shows progressive reduction in dataset size as we apply each cleaning step.")

---

## 10. Lab: The Curator

Now it's your turn! You've learned all the pieces — let's put them together into a complete data cleaning pipeline.

**Your mission:** Create a reusable function `clean_dataset()` that takes a raw DataFrame and returns cleaned train/test splits.

**Requirements:**
1. Remove duplicates
2. Handle missing values appropriately
3. Standardize formatting
4. Filter invalid data
5. Fix data types
6. Split 80/20 train/test
7. Return both DataFrames

**Bonus challenges:**
- Add a `verbose` parameter to print progress
- Add a `random_state` parameter for reproducibility
- Calculate and print statistics about cleaning impact
- Handle edge cases (empty DataFrame, all missing values, etc.)

In [ ]:
def clean_dataset(df_raw, random_state=42, verbose=True):
    """
    Complete data cleaning pipeline.
    
    Parameters:
    -----------
    df_raw : pd.DataFrame
        Raw input DataFrame
    random_state : int
        Random seed for reproducibility
    verbose : bool
        Print progress messages
    
    Returns:
    --------
    train_df, test_df : tuple of pd.DataFrame
        Cleaned training and test DataFrames
    """
    if verbose:
        print("🧹 Starting data cleaning pipeline...\n")
        print(f"Input: {len(df_raw)} rows × {len(df_raw.columns)} columns")
    
    df = df_raw.copy()
    
    # Step 1: Remove duplicates
    if verbose:
        print("\n[1/7] Removing duplicates...")
    before = len(df)
    df = df.drop_duplicates(keep='first')
    if verbose:
        print(f"  Removed {before - len(df)} duplicate rows")
    
    # Step 2: Handle missing values
    if verbose:
        print("\n[2/7] Handling missing values...")
    
    # Drop rows with missing critical columns
    df = df.dropna(subset=['prompt_text'])
    if verbose:
        print(f"  Dropped rows with missing prompt_text")
    
    # Fill missing response_text
    df['response_text'] = df['response_text'].fillna('[No response generated]')
    
    # Fill missing tokens with 0
    df['tokens_used'] = df['tokens_used'].fillna(0)
    
    # Fill missing rating with median
    df['rating'] = df['rating'].fillna(df['rating'].median())
    
    if verbose:
        print(f"  Filled missing values appropriately")
    
    # Step 3: Fix formatting
    if verbose:
        print("\n[3/7] Standardizing formatting...")
    
    df['model'] = df['model'].str.strip().str.lower()
    df['category'] = df['category'].str.strip().str.lower()
    
    # Standardize model names
    model_mapping = {
        'gpt4': 'gpt-4',
        'claude3': 'claude-3'
    }
    df['model'] = df['model'].replace(model_mapping)
    
    if verbose:
        print(f"  Standardized {len(df.columns)} text columns")
    
    # Step 4: Filter invalid data
    if verbose:
        print("\n[4/7] Filtering invalid data...")
    
    before = len(df)
    df = df[df['temperature'] <= 1.0]
    df = df[df['tokens_used'] > 0]
    
    if verbose:
        print(f"  Removed {before - len(df)} rows with invalid values")
    
    # Step 5: Fix data types
    if verbose:
        print("\n[5/7] Correcting data types...")
    
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['tokens_used'] = df['tokens_used'].astype(int)
    
    if verbose:
        print(f"  Converted date to datetime")
        print(f"  Converted tokens_used to int")
    
    # Step 6: Shuffle
    if verbose:
        print("\n[6/7] Shuffling data...")
    
    df = df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    
    # Step 7: Train/test split
    if verbose:
        print("\n[7/7] Splitting train/test (80/20)...")
    
    train_size = int(0.8 * len(df))
    train_df = df.iloc[:train_size]
    test_df = df.iloc[train_size:]
    
    if verbose:
        print(f"  Training: {len(train_df)} rows")
        print(f"  Test: {len(test_df)} rows")
        print("\n✅ Cleaning pipeline complete!")
        print(f"\nSummary: {len(df_raw)} → {len(df)} rows ({(1 - len(df)/len(df_raw))*100:.1f}% reduction)")
    
    return train_df, test_df

In [ ]:
# Test the pipeline
df_raw = pd.read_csv('data/dirty_dataset.csv')
train, test = clean_dataset(df_raw, random_state=42, verbose=True)

In [ ]:
# Verify outputs
print("\n=== Training Set ===")
print(train.info())
print("\n=== Test Set ===")
print(test.info())

### Try It Yourself:

1. **Modify the threshold:** Change the train/test split to 70/30 or 90/10
2. **Add validation set:** Split into train/validation/test (60/20/20)
3. **Different strategy:** Instead of dropping rows with invalid temperature, try capping at 1.0
4. **Add logging:** Make the function save a log file with statistics
5. **Error handling:** Add try/except blocks to handle unexpected data formats

In [ ]:
# Your code here!
# Try implementing one of the challenges above



---

## Summary & Key Takeaways

**What You've Learned:**

1. **Data cleaning is critical** — 80% of ML work, yet often underestimated by beginners

2. **The inspection-first approach** — Always understand your data before cleaning it

3. **Duplication is dangerous** — Causes data leakage and biased models

4. **Missing values require strategy** — Drop vs. fill depends on context, not rules

5. **Boolean indexing is powerful** — Flexible filtering with logical operators

6. **Standardization reduces noise** — Consistent formatting improves model accuracy

7. **Types unlock functionality** — datetime objects enable date math, proper types improve performance

8. **Train/test split is mandatory** — Never test on data you trained on

9. **Pipeline thinking** — Build reusable functions for reproducibility

**The Complete Pipeline:**

```mermaid
flowchart TB
    A[Load Raw Data] --> B{Inspect}
    B --> C[Remove Duplicates]
    C --> D[Handle Missing Values]
    D --> E[Fix Formatting]
    E --> F[Filter Invalid Data]
    F --> G[Correct Data Types]
    G --> H[Shuffle]
    H --> I[Split Train/Test]
    I --> J[Export Clean Data]
    J --> K[Ready for ML]
    
    style A fill:#f9e79f
    style B fill:#fadbd8
    style K fill:#abebc6
```

**Best Practices:**

- ✅ Always make a copy before modifying data
- ✅ Document your cleaning decisions
- ✅ Set random seeds for reproducibility
- ✅ Check distributions before and after cleaning
- ✅ Save intermediate results for debugging
- ✅ Build reusable pipelines

**Common Pitfalls:**

- ❌ Cleaning test data differently than training data
- ❌ Filling missing values with information from the future
- ❌ Dropping too much data without justification
- ❌ Forgetting to set random_state (non-reproducible results)
- ❌ Assuming clean data without inspection

---

**Next Steps:**

In the next module, we'll use this clean data to build our first AI retrieval system. But remember: **the quality of your results depends entirely on the quality of your data.** Master cleaning, and you've mastered half of data science.

> **Final Insight:** "Garbage in, garbage out" isn't just a warning — it's a promise. Clean data is the foundation of reliable AI.